# Supermarket Pricing Dashboard — Initial Analysis

This notebook performs an initial exploratory analysis of the supermarket pricing dataset.

## Goals
- Load and inspect the dataset
- Check data quality and coverage
- Compare prices across chains and categories
- Explore private label vs branded products
- Identify outliers
- Review cost-per-use metrics where available

## 1. Imports

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load data

In [46]:
df = pd.read_excel("Dataset.xlsx")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,product_id,product_name,package_size,brand,private_label,comparability_flag,comparability_note,category,chain,price_eur,unit,price_per_unit,display_unit,uses,cost_per_use,date,data_source
0,dairy_milk_whole_1L,Leche entera,1L,Hacendado,yes,comparable,NaN,dairy,Mercadona,0.96,L,0.960000,€/L,NaN,NaN,2026-04-19,website
1,dairy_milk_semi_1L,Leche semidesnatada,1L,Hacendado,yes,comparable,NaN,dairy,Mercadona,0.84,L,0.840000,€/L,NaN,NaN,2026-04-19,website
2,dairy_yogurt_greek_1kg,Yogur griego natural,1kg,Hacendado,yes,comparable,NaN,dairy,Mercadona,2.20,kg,2.200000,€/kg,NaN,NaN,2026-04-19,website
3,dairy_yogurt_natural_pack_125g,Yogur natural,6x125g,Hacendado,yes,comparable,NaN,dairy,Mercadona,1.00,kg,1.333333,€/kg,NaN,NaN,2026-04-19,website
4,dairy_yogurt_natural_pack_120g_branded,Yogur natural Danone,8x120g,Danone,no,comparable,NaN,dairy,Mercadona,1.99,kg,2.072917,€/kg,NaN,NaN,2026-04-19,website


## 3. Data quality checks

In [47]:
print("Shape:", df.shape)

# Missing values
display(df.isna().sum().sort_values(ascending=False))

# Duplicates
print("Duplicates:", df.duplicated().sum())

# Coverage per product
coverage = df.groupby("product_id")["chain"].nunique()
display(coverage[coverage < 5])

Shape: (299, 17)


comparability_note    289
uses                  251
cost_per_use          251
product_id              0
product_name            0
private_label           0
comparability_flag      0
brand                   0
package_size            0
chain                   0
category                0
price_eur               0
unit                    0
display_unit            0
price_per_unit          0
date                    0
data_source             0
dtype: int64

Duplicates: 0


product_id
breakfast_cereals_cornflakes_500g_branded         4
breakfast_juice_orange_not_from_concentrate_1L    4
cleaning_dishwasher_tablets_approx_30u_branded    4
fresh_eggs_size_M_12u                             4
snacks_crisps_salted_248g_branded                 4
Name: chain, dtype: int64

## 4. Descriptive overview

In [48]:
display(df.describe())

display(
    df.groupby("category")["price_per_unit"]
    .mean()
    .sort_values()
)

,price_eur,price_per_unit,uses,cost_per_use,date
count,299.000000,299.000000,48.000000,48.000000,299
mean,2.625184,4.287988,36.375000,0.152549,2026-04-25 23:26:17.257525
min,0.650000,0.086250,20.000000,0.067115,2026-04-19 00:00:00
25%,1.500000,0.900000,30.000000,0.094318,2026-04-19 00:00:00
50%,2.000000,2.200000,36.000000,0.101944,2026-04-19 00:00:00
75%,3.400000,6.746667,44.250000,0.183333,2026-05-03 00:00:00
max,18.750000,19.600000,55.000000,0.625000,2026-05-03 00:00:00
std,1.949614,4.361498,9.173563,0.117425,NaN


category
cleaning         1.471413
fresh produce    1.976672
dairy            3.331613
breakfast        4.556157
snacks           8.597910
Name: price_per_unit, dtype: float64

## 5. Price index

In [49]:
# Relative to market average = 100

overall_avg = df["price_per_unit"].mean()

price_index = (
    df.groupby("chain")["price_per_unit"]
    .mean()
    .reset_index()
)

price_index["price_index"] = (price_index["price_per_unit"] / overall_avg) * 100

display(price_index.round(2))

,chain,price_per_unit,price_index
0,Alcampo,4.43,103.32
1,Carrefour,4.47,104.29
2,Consum,4.22,98.53
3,Dia,4.30,100.25
4,Mercadona,4.03,93.96


## 6. Cheapest chain per category

In [50]:
avg_price_category_chain = (
    df.groupby(["category", "chain"])["price_per_unit"]
    .mean()
    .reset_index()
)

cheapest = avg_price_category_chain.loc[
    avg_price_category_chain.groupby("category")["price_per_unit"].idxmin()
]

display(cheapest)

,category,chain,price_per_unit
0,breakfast,Alcampo,4.110694
9,cleaning,Mercadona,1.274762
14,dairy,Mercadona,3.191220
19,fresh produce,Mercadona,1.803667
24,snacks,Mercadona,8.104398


## 7. Private label vs branded

In [51]:
pl_vs_brand = (
    df.groupby("private_label")["price_per_unit"]
    .agg(["count", "mean"])
    .reset_index()
)

display(pl_vs_brand.round(3))

,private_label,count,mean
0,no,66,3.747
1,yes,233,4.441


## 8. Cost-per-use analysis
Only applicable to products where `cost_per_use` is available.

In [52]:
if "cost_per_use" in df.columns and df["cost_per_use"].notna().any():
    
    cleaning_use = df[df["cost_per_use"].notna()].copy()

    display(
        cleaning_use[
            ["product_id", "chain", "brand", "price_eur", "price_per_unit", "cost_per_use"]
        ].sort_values(["product_id", "chain"]).round(3)
    )

    cost_use_summary = (
        cleaning_use
        .groupby(["product_id", "chain"])["cost_per_use"]
        .mean()
        .reset_index()
    )

    display(cost_use_summary.sort_values(["product_id", "cost_per_use"]).round(3))

else:
    print("No valid cost_per_use data available.")

,product_id,chain,brand,price_eur,price_per_unit,cost_per_use
135,cleaning_dishwasher_gel_approx_750ml,Alcampo,Auchan,3.42,4.750,0.095
285,cleaning_dishwasher_gel_approx_750ml,Alcampo,Auchan,3.42,4.750,0.095
76,cleaning_dishwasher_gel_approx_750ml,Carrefour,Carrefour Expert,4.25,4.722,0.094
226,cleaning_dishwasher_gel_approx_750ml,Carrefour,Carrefour Expert,4.25,4.722,0.094
46,cleaning_dishwasher_gel_approx_750ml,Consum,Consum,3.35,4.653,0.093
196,cleaning_dishwasher_gel_approx_750ml,Consum,Consum,3.45,4.792,0.096
106,cleaning_dishwasher_gel_approx_750ml,Dia,Dia Super Paco,4.09,6.312,0.114
256,cleaning_dishwasher_gel_approx_750ml,Dia,Dia Super Paco,4.09,6.312,0.114
16,cleaning_dishwasher_gel_approx_750ml,Mercadona,Bosque Verde,3.45,4.792,0.096
166,cleaning_dishwasher_gel_approx_750ml,Mercadona,Bosque Verde,3.45,4.792,0.096


,product_id,chain,cost_per_use
1,cleaning_dishwasher_gel_approx_750ml,Carrefour,0.094
2,cleaning_dishwasher_gel_approx_750ml,Consum,0.094
0,cleaning_dishwasher_gel_approx_750ml,Alcampo,0.095
4,cleaning_dishwasher_gel_approx_750ml,Mercadona,0.096
3,cleaning_dishwasher_gel_approx_750ml,Dia,0.114
8,cleaning_dishwasher_tablets_approx_30u,Dia,0.086
7,cleaning_dishwasher_tablets_approx_30u,Consum,0.098
9,cleaning_dishwasher_tablets_approx_30u,Mercadona,0.098
5,cleaning_dishwasher_tablets_approx_30u,Alcampo,0.106
6,cleaning_dishwasher_tablets_approx_30u,Carrefour,0.109


## 9. Temporal comparison

In [ ]:
# Compare prices per unit between dates

if "date" in df.columns:

    pivot_time = df.pivot_table(
        index=["product_id", "chain"],
        columns="date",
        values="price_per_unit"
    )

    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    pivot_time["change"] = pivot_time.iloc[:, -1] - pivot_time.iloc[:, 0]

    display(pivot_time.sort_values("change", ascending=False).head(10).round(3))

date                                                  2026-04-19  2026-05-03  \
product_id                                 chain                               
breakfast_coffee_ground_approx_250g        Carrefour       9.880      19.560   
                                           Mercadona      10.400      19.600   
breakfast_cereals_chocolate_approx_500g    Consum          6.224       7.344   
breakfast_cereals_cornflakes_500g_branded  Mercadona       5.000       6.000   
breakfast_juice_orange_from_concentrate_1L Carrefour       1.850       2.090   
cleaning_laundry_liquid_approx_3L          Consum          1.342       1.573   
cleaning_dishwasher_gel_approx_750ml       Consum          4.653       4.792   
dairy_cheese_sliced_approx_200g            Dia             7.167       7.292   
fresh_eggs_size_L_12u                      Alcampo         0.189       0.265   
breakfast_coffee_capsules_with_milk_16     Alcampo         0.221       0.294   

date                                                  change  
product_id                                 chain              
breakfast_coffee_ground_approx_250g        Carrefour   9.680  
                                           Mercadona   9.200  
breakfast_cereals_chocolate_approx_500g    Consum      1.120  
breakfast_cereals_cornflakes_500g_branded  Mercadona   1.000  
breakfast_juice_orange_from_concentrate_1L Carrefour   0.240  
cleaning_laundry_liquid_approx_3L          Consum      0.231  
cleaning_dishwasher_gel_approx_750ml       Consum      0.139  
dairy_cheese_sliced_approx_200g            Dia         0.125  
fresh_eggs_size_L_12u                      Alcampo     0.076  
breakfast_coffee_capsules_with_milk_16     Alcampo     0.073

## 10. Basket simulation

In [67]:
# Fixed basket: one of each product

basket = (
    df.groupby(["chain", "product_id"])["price_eur"]
    .mean()
    .reset_index()
)

basket_total = (
    basket.groupby("chain")["price_eur"]
    .sum()
    .sort_values()
)

display(basket_total)

chain
Alcampo      67.530
Mercadona    74.275
Consum       79.040
Dia          83.985
Carrefour    89.730
Name: price_eur, dtype: float64

## 11. Outlier review

In [68]:
display(
    df.sort_values("price_per_unit", ascending=False)
    .head(10)
)

display(
    df.sort_values("price_per_unit", ascending=True)
    .head(10)
)

,product_id,product_name,package_size,brand,private_label,comparability_flag,comparability_note,category,chain,price_eur,unit,price_per_unit,display_unit,uses,cost_per_use,date,data_source
162,breakfast_coffee_ground_approx_250g,Café molido natural,250g,Hacendado,yes,comparable,NaN,breakfast,Mercadona,4.90,kg,19.60,€/kg,NaN,NaN,2026-05-03,website
222,breakfast_coffee_ground_approx_250g,Café molido natural,250g,Carrefour Classic,yes,comparable,NaN,breakfast,Carrefour,4.89,kg,19.56,€/kg,NaN,NaN,2026-05-03,website
120,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Dia Temptation,yes,comparable,NaN,snacks,Dia,1.95,kg,19.50,€/kg,NaN,NaN,2026-04-19,website
89,snacks_chocolate_dark_85_100g,Chocolate negro intenso 85% cacao sin gluten,100g,Carrefour Selection,yes,comparable,NaN,snacks,Carrefour,1.95,kg,19.50,€/kg,NaN,NaN,2026-04-19,website
239,snacks_chocolate_dark_85_100g,Chocolate negro intenso 85% cacao sin gluten,100g,Carrefour Selection,yes,comparable,NaN,snacks,Carrefour,1.91,kg,19.10,€/kg,NaN,NaN,2026-05-03,website
270,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Dia Temptation,yes,comparable,NaN,snacks,Dia,1.89,kg,18.90,€/kg,NaN,NaN,2026-05-03,website
29,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Hacendado,yes,comparable,NaN,snacks,Mercadona,1.75,kg,17.50,€/kg,NaN,NaN,2026-04-19,website
60,snacks_chocolate_dark_85_100g,Chocolate Negro 85% Cacao,100g,Consum,yes,comparable,NaN,snacks,Consum,1.75,kg,17.50,€/kg,NaN,NaN,2026-04-19,website
210,snacks_chocolate_dark_85_100g,Chocolate Negro 85% Cacao,100g,Consum,yes,comparable,NaN,snacks,Consum,1.75,kg,17.50,€/kg,NaN,NaN,2026-05-03,website
179,snacks_chocolate_dark_85_100g,Chocolate negro 85% cacao,100g,Hacendado,yes,comparable,NaN,snacks,Mercadona,1.75,kg,17.50,€/kg,NaN,NaN,2026-05-03,website


,product_id,product_name,package_size,brand,private_label,comparability_flag,comparability_note,category,chain,price_eur,unit,price_per_unit,display_unit,uses,cost_per_use,date,data_source
254,cleaning_dishwasher_tablets_approx_30u,Pastillas lavavajillas,40u,Dia Super Paco,yes,comparable,NaN,cleaning,Dia,3.45,unit,0.086250,€/unit,40.0,0.086250,2026-05-03,website
104,cleaning_dishwasher_tablets_approx_30u,Pastillas lavavajillas,40u,Dia Super Paco,yes,comparable,NaN,cleaning,Dia,3.45,unit,0.086250,€/unit,40.0,0.086250,2026-04-19,website
14,cleaning_dishwasher_tablets_approx_30u,Lavavajillas Todo en 1 en pastillas,30u,Bosque Verde,yes,comparable,NaN,cleaning,Mercadona,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-04-19,website
44,cleaning_dishwasher_tablets_approx_30u,Pastillas Lavavajillas Todo en 1,30u,Consum,yes,comparable,NaN,cleaning,Consum,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-04-19,website
194,cleaning_dishwasher_tablets_approx_30u,Pastillas Lavavajillas Todo en 1,30u,Consum,yes,comparable,NaN,cleaning,Consum,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-05-03,website
164,cleaning_dishwasher_tablets_approx_30u,Lavavajillas Todo en 1 en pastillas,30u,Bosque Verde,yes,comparable,NaN,cleaning,Mercadona,2.95,unit,0.098333,€/unit,30.0,0.098333,2026-05-03,website
134,cleaning_dishwasher_tablets_approx_30u,"Detergente para lavavajillas para máquinas, fr...",45u,Auchan,yes,comparable,NaN,cleaning,Alcampo,4.75,unit,0.105556,€/unit,45.0,0.105556,2026-04-19,website
284,cleaning_dishwasher_tablets_approx_30u,"Detergente para lavavajillas para máquinas, fr...",45u,Auchan,yes,comparable,NaN,cleaning,Alcampo,4.75,unit,0.105556,€/unit,45.0,0.105556,2026-05-03,website
224,cleaning_dishwasher_tablets_approx_30u,Lavavajillas a máquina en pastillas,40u,Carrefour Expert,yes,comparable,NaN,cleaning,Carrefour,4.35,unit,0.108750,€/unit,40.0,0.108750,2026-05-03,website
74,cleaning_dishwasher_tablets_approx_30u,Lavavajillas a máquina en pastillas,40u,Carrefour Expert,yes,comparable,NaN,cleaning,Carrefour,4.35,unit,0.108750,€/unit,40.0,0.108750,2026-04-19,website


## 12. Insight summary

### Key Findings
1. Price differences across categories are larger than differences across chains, suggesting that product mix has a stronger impact on total spending than retailer choice.
2. Private label products consistently undercut branded alternatives, reinforcing their role as a key competitive pricing lever.
3. No single chain is the cheapest across all categories, indicating targeted pricing strategies rather than uniform positioning.
4. Initial temporal analysis suggests moderate price stability across most categories, with some variation in cleaning products.
5. Basket-level comparison shows that total cost differences between chains are narrower than expected, despite product-level variation.